In [16]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/wcukierski/enron-email-dataset/emails.csv


# Enron received-mail sampling for a junk-mail filter
Goal: 500+ labelled received emails. Source: wcukierski/enron-email-dataset. Kaggle: rehanmhate

## Step 0 – Find the data path

In [17]:
import os
for root, _, files in os.walk('/kaggle/input'):
    print(root, files)

/kaggle/input []
/kaggle/input/datasets []
/kaggle/input/datasets/wcukierski []
/kaggle/input/datasets/wcukierski/enron-email-dataset ['emails.csv']


## Step 1 – Load raw emails

In [18]:
import pandas as pd, email
PATH = '/kaggle/input/datasets/wcukierski/enron-email-dataset/emails.csv'
df = pd.read_csv(PATH)
print(df.shape); df.head(3)

(517401, 2)


,file,message
0,allen-p/_sent_mail/1.,"Message-ID: <18782981.1075855378110.JavaMail.evans@thyme>\nDate: Mon, 14 May..."
1,allen-p/_sent_mail/10.,"Message-ID: <15464986.1075855378456.JavaMail.evans@thyme>\nDate: Fri, 4 May ..."
2,allen-p/_sent_mail/100.,"Message-ID: <24216240.1075855687451.JavaMail.evans@thyme>\nDate: Wed, 18 Oct..."


## Step 2 – Build key, mailbox, folder; drop sent folders

In [19]:
df['key'] = df['file'].str.rstrip('.')          # e.g. allen-p/inbox/12
parts = df['key'].str.split('/')
df['mailbox'] = parts.str[0]
df['folder'] = parts.str[1:-1].str.join('/')

SENT = {'sent', 'sent_items', '_sent_mail'}
df['is_sent'] = parts.apply(lambda p: any(x in SENT for x in p[1:-1]))
print(df['is_sent'].value_counts())

recv = df[~df['is_sent']].copy()
print(len(recv), 'received emails |', recv['mailbox'].nunique(), 'mailboxes')

is_sent
False    391075
True     126326
Name: count, dtype: int64
391075 received emails | 150 mailboxes


## Step 3 – Parse subject, from, to, body

In [20]:
def parse(raw):
    m = email.message_from_string(raw)
    b = m.get_payload()
    return pd.Series({
        'subject': m.get('Subject', '') or '',
        'from_address': m.get('From', '') or '',
        'to_addresses': m.get('To', '') or '',
        'body': b if isinstance(b, str) else '',
    })

recv = recv.join(recv['message'].apply(parse))
print(recv.shape)
recv[['key', 'subject', 'from_address', 'to_addresses']].head()

(391075, 10)


,key,subject,from_address,to_addresses
602,allen-p/all_documents/1,"December 14, 2000 - Bear Stearns' predictions for telecom in Latin\n America",1.11913372.-2@multexinvestornetwork.com,pallen@enron.com
603,allen-p/all_documents/10,Bloomberg Power Lines Report,messenger@ecm.bloomberg.com,
604,allen-p/all_documents/100,Consolidated positions: Issues & To Do list,phillip.allen@enron.com,keith.holst@enron.com
605,allen-p/all_documents/101,Consolidated positions: Issues & To Do list,phillip.allen@enron.com,keith.holst@enron.com
606,allen-p/all_documents/102,,phillip.allen@enron.com,david.delainey@enron.com


## Step 4 – Spread across folders and mailboxes

In [21]:
print('Top folders:')
print(recv['folder'].value_counts().head(25))
print('\nEmails per mailbox:')
print(recv.groupby('mailbox').size().describe())

Top folders:
folder
all_documents                            128103
discussion_threads                        58609
deleted_items                             50991
inbox                                     41509
notes_inbox                               36665
calendar/untitled                          4478
archiving/untitled                         4477
personal                                   2420
attachments                                2026
meetings                                   1872
calendar                                   1655
schedule_crawler                           1398
logistics                                  1192
tw_commercial_group                        1159
california                                 1052
bill_williams_iii                          1022
deal_communication/deal_discrepancies       897
_americas/esvl                              715
e_mail_bin                                  614
tufco                                       609
resumes             

## Step 5 – Remove copies of the owner's own sent mail
Folders like all_documents and discussion_threads also hold mail the owner sent. Owner address = sender(s) making up >=10% of that mailbox's sent folders. Rows from those addresses are removed.

In [22]:
import re
sent = df[df['is_sent']]
sent_from = sent['message'].str.extract(r'^From:\s*(.+)$', flags=re.M)[0].str.strip().str.lower()
sent_from.name = 'addr'
share = sent_from.groupby(sent['mailbox']).value_counts(normalize=True)
owner = share[share >= 0.10].reset_index().groupby('mailbox')['addr'].apply(set).to_dict()
print('mailboxes with known owner address:', len(owner), 'of', recv['mailbox'].nunique())

recv['from_l'] = recv['from_address'].str.strip().str.lower()
recv['own_sent'] = [f in owner.get(mb, set()) for f, mb in zip(recv['from_l'], recv['mailbox'])]
print(recv['own_sent'].value_counts())
print(recv.loc[recv['own_sent'], 'folder'].value_counts().head(10))
recv = recv[~recv['own_sent']].copy()
print(len(recv), 'received emails left')

mailboxes with known owner address: 149 of 150
own_sent
False    301030
True      90045
Name: count, dtype: int64
folder
all_documents         58755
discussion_threads    21452
deleted_items          2830
calendar/untitled      1307
archiving/untitled     1307
attachments             436
contacts                278
_sent                   250
calendar                194
to_do                   189
Name: count, dtype: int64
301030 received emails left


## Step 6 – Clean: leftover sent folders, empty bodies, exact duplicates
- Drop any folder that is a sent folder under another name (e.g. `_sent`).
- Drop emails with no body text.
- Drop exact duplicates (same sender, subject and body), keeping the first copy.

In [23]:
print('rows in:', len(recv))

# 1. leftover sent folders under other names (e.g. _sent)
SENT_RE = r'(?:^|/)_?sent(?:_items|_mail)?(?:/|$)'
before = len(recv)
print(recv.loc[recv['folder'].str.contains(SENT_RE), 'folder'].value_counts())
recv = recv[~recv['folder'].str.contains(SENT_RE)].copy()
print('leftover sent-folder rows dropped:', before - len(recv))

# 2. empty bodies
recv['body'] = recv['body'].str.strip()
before = len(recv)
recv = recv[recv['body'].str.len() > 0].copy()
print('empty bodies dropped:', before - len(recv))

# 3. exact duplicates (same sender, subject, body)
before = len(recv)
recv = recv.drop_duplicates(subset=['from_l', 'subject', 'body']).copy()
print('exact duplicates dropped:', before - len(recv))

print(len(recv), 'left |', recv['mailbox'].nunique(), 'mailboxes')

rows in: 301030
folder
_sent    15
Name: count, dtype: int64
leftover sent-folder rows dropped: 15
empty bodies dropped: 0
exact duplicates dropped: 147593
153422 left | 150 mailboxes


## Step 7 – Spread after cleaning
Check the cleaned pool still covers many mailboxes and folders.

In [24]:
print('Top folders:')
print(recv['folder'].value_counts().head(15))
print('\nEmails per mailbox:')
print(recv.groupby('mailbox').size().describe())
print('\nSmallest mailboxes:')
print(recv.groupby('mailbox').size().sort_values().head(5))

Top folders:
folder
all_documents          61218
deleted_items          36825
inbox                  30017
meetings                1779
calendar                1376
personal                 855
bill_williams_iii        715
_americas/esvl           634
schedule_crawler         519
tw_commercial_group      449
inbox/archives           438
inbox/junk_file          427
inbox/junk               423
e_mail_bin               359
eesirenewableenergy      295
Name: count, dtype: int64

Emails per mailbox:
count      150.000000
mean      1022.813333
std       1321.718580
min         10.000000
25%        346.750000
50%        658.000000
75%       1168.750000
max      10327.000000
dtype: float64

Smallest mailboxes:
mailbox
dickson-s    10
merriss-s    11
phanis-s     18
south-s      32
ring-a       48
dtype: int64


## Step 8 – Keyword pools to find rare categories
Pools are search helpers only, NOT labels. Subject hints from the task guide ("Out of Office", "Undeliverable", "Accepted:", "Invitation:"), plus sender domains. Every email is labelled by hand later.

In [25]:
subj = recv['subject'].str.lower()
frm = recv['from_l']
fold = recv['folder'].str.lower()

PERSONAL_DOM = r'@(?:aol|hotmail|yahoo|msn|earthlink|juno|excite|mindspring|swbell|prodigy|att|home|pdq)\.'
external = ~frm.str.contains(r'@enron\.com') & frm.str.contains('@')

pools = {
    'auto_reply': subj.str.contains(r'out of (?:the )?office|auto(?:matic)?[- ]?reply|away from (?:my|the) office|on vacation|autoreply'),
    'bounce':     frm.str.contains(r'mailer-daemon|postmaster') |
                  subj.str.contains(r'undeliverable|delivery (?:status notification|failure|failed)|returned mail|mail delivery failed|could not be delivered'),
    'calendar':   subj.str.contains(r'^(?:accepted|declined|tentative|updated|invitation|cancelled|canceled)\s*:') |
                  subj.str.contains(r'meeting request'),
    'workflow':   frm.str.contains(r'arsystem|workflow|ehronline|sap|noreply|no-reply|itcentral|ibuyit|timesheet') |
                  subj.str.contains(r'approval|has been (?:approved|submitted|rejected)|action required|request #|your request'),
    'personal':   frm.str.contains(PERSONAL_DOM) | fold.str.contains('personal'),
    'external_biz': external & ~frm.str.contains(PERSONAL_DOM),
    'junk_folder':  fold.str.contains('junk'),
}
for k, v in pools.items():
    recv['pool_' + k] = v.values
    print(f'{k:13s} {int(v.sum()):7d} emails | {recv.loc[v.values, "mailbox"].nunique():3d} mailboxes')

auto_reply        632 emails |  96 mailboxes
bounce            635 emails |  97 mailboxes
calendar          394 emails |  74 mailboxes
workflow         3415 emails | 117 mailboxes
personal         7252 emails | 144 mailboxes
external_biz    49366 emails | 149 mailboxes
junk_folder       858 emails |   4 mailboxes


## Step 9 – Sample ~600 candidates across many mailboxes
Pools are sampled in order, without repeats, with a cap per mailbox per pool, random seed 42.
Quotas are above 30 because keyword pools contain false hits; the final category is decided by hand.
The rest is a random sample of the whole clean pool (mostly colleague mail, capped later at 200).

In [26]:
SEED = 42
QUOTAS = [  # (pool, how many, max per mailbox)
    ('auto_reply',   50, 3),
    ('bounce',       50, 3),
    ('calendar',     50, 3),
    ('workflow',     60, 3),
    ('personal',     60, 3),
    ('external_biz', 120, 3),
    ('junk_folder',  25, 7),
]
RANDOM_N, RANDOM_CAP = 185, 2

def pick(cand, n, cap):
    cand = cand.sample(frac=1, random_state=SEED)
    cand = cand[cand.groupby('mailbox').cumcount() < cap]
    return cand.head(n)

parts, taken = [], set()
for pool, n, cap in QUOTAS:
    cand = recv[recv['pool_' + pool] & ~recv.index.isin(taken)]
    got = pick(cand, n, cap).assign(picked_from=pool)
    taken |= set(got.index); parts.append(got)
    print(f'{pool:13s} wanted {n:3d} got {len(got):3d} from {got["mailbox"].nunique():3d} mailboxes')

rest = recv[~recv.index.isin(taken)]
got = pick(rest, RANDOM_N, RANDOM_CAP).assign(picked_from='random')
parts.append(got)
print(f'{"random":13s} wanted {RANDOM_N:3d} got {len(got):3d} from {got["mailbox"].nunique():3d} mailboxes')

sample = pd.concat(parts)
print('\nsample:', len(sample), 'emails |', sample['mailbox'].nunique(), 'mailboxes | max per mailbox:',
      sample.groupby('mailbox').size().max())

auto_reply    wanted  50 got  50 from  35 mailboxes
bounce        wanted  50 got  50 from  34 mailboxes
calendar      wanted  50 got  50 from  34 mailboxes
workflow      wanted  60 got  60 from  42 mailboxes
personal      wanted  60 got  60 from  41 mailboxes
external_biz  wanted 120 got 120 from  73 mailboxes
junk_folder   wanted  25 got  22 from   4 mailboxes
random        wanted 185 got 185 from 110 mailboxes

sample: 597 emails | 135 mailboxes | max per mailbox: 16


## Step 10 – English check (flag, then review)
langdetect on subject + first 1,000 characters of body, seed fixed. Flagged emails are listed and checked by eye before dropping, because short texts can be misdetected.

In [27]:
try:
    from langdetect import detect, DetectorFactory
except ImportError:
    !pip install -q langdetect
    from langdetect import detect, DetectorFactory
DetectorFactory.seed = 0

def lang(row):
    try:
        return detect((row['subject'] + '\n' + row['body'][:1000]).strip())
    except Exception:
        return 'unknown'

sample['lang'] = sample.apply(lang, axis=1)
print(sample['lang'].value_counts())

flagged = sample[sample['lang'] != 'en']
pd.set_option('display.max_colwidth', 80)
flagged[['key', 'lang', 'subject']].assign(body_start=flagged['body'].str[:80].str.replace('\n', ' '))

lang
en    591
af      2
fr      2
so      2
Name: count, dtype: int64


,key,lang,subject,body_start
248279,kean-s/enron_advisonry_council/56,af,Howard Davies invite letter,- Davies (Howard) invitation 07-24-01.doc
331600,mckay-j/deleted_items/69,af,Week 10,- JonMCKAY061101.xls
422115,shackleton-s/inbox/865,fr,DRIP list,- DRIPincome2001.doc
507755,williams-j/6day/17,so,,- GC010915.xls
512705,wolfe-j/08_saved/34,so,FW: Too much birthday fun!!!,http://www.ofoto.com/BrowsePhotos.jsp?UV=355983206630_94253538103&US=0&colli...
298122,maggi-m/deleted_items/169,fr,BNP PARIBAS Commodity Futures NG MarketWatch For 01/29/02,(See attached file: g012902.pdf) _________________________________________...


## Step 11 – Drop non-English and attachment-only emails
langdetect flagged 6 of 597; checked by eye: none were really non-English. 5 emails had only an attachment filename as the body (4 caught by the attachment rule, 1 – kean-s/enron_advisonry_council/56 – via the language flag). Rule "skip emails with no body text": all 5 dropped -> 592.

In [28]:
ATTACH_LINE = r'^\s*(?:-\s*)?\S+\.(?:doc|xls|pdf|ppt|txt|zip|jpg|gif|tif|htm|html|csv|wpd|rtf)\s*$'
def real_text(b):
    lines = [l for l in b.splitlines() if l.strip() and not re.match(ATTACH_LINE, l, flags=re.I)]
    return '\n'.join(lines).strip()

sample['attach_only'] = sample['body'].apply(lambda b: real_text(b) == '')
print('attachment-only bodies:', sample['attach_only'].sum())
print(sample.loc[sample['attach_only'], ['key', 'subject']].to_string())

KEEP_EN = {'maggi-m/deleted_items/169', 'wolfe-j/08_saved/34'}   # flagged by langdetect but English (checked by eye)
non_en = (sample['lang'] != 'en') & ~sample['key'].isin(KEEP_EN)
print('non-English dropped (after eye check):', int((non_en & ~sample['attach_only']).sum()))

before = len(sample)
sample = sample[~sample['attach_only'] & ~non_en].copy()
print('dropped', before - len(sample), '->', len(sample), 'left |', sample['mailbox'].nunique(), 'mailboxes')
print(sample['picked_from'].value_counts())

attachment-only bodies: 4
                             key                                    subject
331600  mckay-j/deleted_items/69                                    Week 10
422115    shackleton-s/inbox/865                                  DRIP list
507755        williams-j/6day/17                                           
272019           lay-k/inbox/401  L. Kudlow/J. Park commentary (10/15/2001)
non-English dropped (after eye check): 1
dropped 5 -> 592 left | 135 mailboxes
picked_from
random          184
external_biz    119
workflow         60
personal         57
calendar         50
bounce           50
auto_reply       50
junk_folder      22
Name: count, dtype: int64


## Step 12 – Final columns, save candidates.csv for labelling
- source = dataset name on every row
- from_address = "Name <address>" (name from X-From); to_addresses = "Name <address>; ..." when X-To names line up one-to-one with To addresses, otherwise addresses only
- HTML bodies converted to plain text; quoted replies and signatures kept
- category and unsure_note left blank: filled in by hand

In [29]:
from bs4 import BeautifulSoup
SOURCE = 'wcukierski/enron-email-dataset'

def clean_name(x):
    return re.sub(r'<[^>]*>', '', x or '').strip().strip('"').strip()

def unfold(x):
    return re.sub(r'\s+', ' ', x or '').strip()

def fmt_from(m):
    addr, name = unfold(m.get('From')), clean_name(unfold(m.get('X-From')))
    if not addr: return ''
    return f'{name} <{addr}>' if name and name.lower() != addr.lower() else addr

def fmt_to(m):
    addrs = [a.strip() for a in unfold(m.get('To')).split(',') if a.strip()]
    names = [clean_name(n) for n in re.sub(r'<[^>]*>', '', unfold(m.get('X-To'))).split(',') if n.strip()]
    if addrs and len(names) == len(addrs):
        return '; '.join(f'{n} <{a}>' if n and n.lower() != a.lower() else a for n, a in zip(names, addrs))
    return '; '.join(addrs)

HTML_RE = r'<(?:html|body|p|br|div|table|font)\b'
def to_text(b):
    return BeautifulSoup(b, 'html.parser').get_text('\n').strip() if re.search(HTML_RE, b, re.I) else b

msgs = sample['message'].apply(email.message_from_string)
sample['is_html'] = sample['body'].str.contains(HTML_RE, case=False, regex=True)
print('HTML bodies converted:', int(sample['is_html'].sum()))

out = pd.DataFrame({
    'key': sample['key'],
    'source': SOURCE,
    'subject': sample['subject'].apply(unfold),
    'body': sample['body'].apply(to_text),
    'from_address': msgs.apply(fmt_from),
    'to_addresses': msgs.apply(fmt_to),
    'category': '',
    'unsure_note': '',
    'picked_from': sample['picked_from'],   # helper only, removed in the final emails.csv
})
print('empty to_addresses:', (out['to_addresses'] == '').sum(), '| names paired:', out['to_addresses'].str.contains('<').sum())
out.to_csv('/kaggle/working/candidates.csv', index=False, encoding='utf-8')
print(out.shape)
out[['key', 'from_address', 'to_addresses', 'subject']].head(8)

HTML bodies converted: 13
empty to_addresses: 19 | names paired: 138
(592, 9)


,key,from_address,to_addresses,subject
185109,jones-t/all_documents/4454,Mary Lynne Ruffer <mary.ruffer@enron.com>,Tana Jones <tana.jones@enron.com>,Mary Lynne Ruffer/HOU/ECT is out of the office.
163925,heard-m/deleted_items/230,"Johnson, Cheryl <cheryl.johnson@enron.com>",marie.heard@enron.com,Out of Office AutoReply: Amerada Hess Corporation
454662,steffes-j/inbox/234,JoAnn P Russell <jprussell@duke-energy.com>,el00-95@listserv.gsa.gov,JoAnn P Russell/Hou-Admin/EnergyTrading/PEC is out of the office.
87489,dean-c/inbox/692,"Chatterton, Jill <jill.chatterton@enron.com>",andy.chen@enron.com; timothy.coffing@enron.com; david.frost@enron.com; john....,Out of Office
371009,ring-r/deleted_items/32,"Bass, Kathy <kathy.bass@enron.com>",robert.gerry@enron.com; guy.sharfman@enron.com; john.zurita@enron.com; heath...,out of the office at 3pm
286966,lokay-m/deleted_items/284,"Robertson, Audrey <audrey.robertson@enron.com>",michelle.lokay@enron.com,Out of Office AutoReply: Aquila Red Lake/TW strategy meeting
94272,donoho-l/inbox/56,"Powers, Ken <ken.powers@enron.com>",lindy.donoho@enron.com; jeanette.doll@enron.com,FW: Out of Office
39066,buy-r/inbox/676,"Port, David <david.port@enron.com>",rick.buy@enron.com,Out of Office AutoReply: Pug's Fax
